# Overfitting y Train-Test Split

## Configuración del Entorno

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def plot_regression(X, y, X_test=None, y_test=None, model=None):
    plt.scatter(X, y, color='blue', label='Train')
    if X_test is not None:
        plt.scatter(X_test, y_test, color='orange', label='Test')
    X_plot = pd.DataFrame(np.linspace(X.min(), X.max(), num=100), columns=X.columns)
    if model is not None:
        plt.plot(X_plot, model.predict(X_plot), color='red', label='Modelo')
    plt.xlabel("Años de Experiencia")
    plt.ylabel("Salario")
    if X_test is not None or model is not None:
        plt.legend()
    plt.show()

## Definición del Problema y Preparación de los Datos

Retomando los ejemplos de los notebooks de regresión lineal simple y decision tree, nos centraremos en el concepto de **overfitting**, enfatizando la importancia de dividir los datos en conjuntos de training y test.

Habitualmente, se separa una parte de los datos para usarla como **conjunto de test**, mientras que el modelo se entrena con el resto (**conjunto de train**). Esto evita el **overfitting**, que ocurre cuando el modelo se ajusta demasiado a los datos de training y no generaliza bien a datos nuevos. Si el error en el conjunto de training es bajo pero alto en el de test, el modelo tiene overfitting.

Es habitual usar el 20% de los datos para el conjunto de test, aunque esto depende del tamaño del dataset. Cuanto mayor sea el dataset, menos datos necesitamos para el conjunto de test.

Usaremos el dataset de predicción de salarios basado en años de experiencia laboral.

In [ ]:
df = pd.read_csv("data/salaries2.csv")

X = df[['YearsExperience']]
y = df['Salary']

plot_regression(X, y)

y definimos el Error Absoluto Medio (MAE) como la métrica de rendimiento.

In [ ]:
from sklearn.metrics import mean_absolute_error

Dividimos los datos en dos conjuntos: training y test. **Entrenamos los modelos con los datos de training y evaluamos su rendimiento en el conjunto de test**.

El parámetro `random_state=42` garantiza la reproducibilidad: ejecutar el código varias veces producirá la misma división.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Regresión Lineal

Entrenamos un modelo de regresión lineal con los datos de training.

In [ ]:
from sklearn.linear_model import LinearRegression
model_linear = LinearRegression().fit(X_train, y_train)

Podemos ver la representación gráfica de la regresión lineal en rojo, calculada con los datos de training (muestras azules) y los datos de test en naranja (que se usan para evaluar el rendimiento del modelo).

In [ ]:
plot_regression(X_train, y_train, X_test, y_test, model_linear)

y calculamos el rendimiento del modelo **en el conjunto de test**: cuánto se desvían las predicciones del modelo (entrenado con el conjunto de training) de los valores reales.

In [ ]:
y_test_pred = model_linear.predict(X_test)
mae_linear = mean_absolute_error(y_test, y_test_pred)
print("MAE Linear Regression: ", mae_linear)

Vemos que las predicciones del modelo de regresión lineal se desvían en promedio $14,907 anuales de los salarios reales.

Si hubiéramos calculado la desviación con el conjunto de training (aunque esto es generalmente un error), el error no sería muy diferente en este caso.

In [ ]:
mae_linear_train = mean_absolute_error(y_train, model_linear.predict(X_train))
print("MAE Linear Regression (train): ", mae_linear_train)

## Árbol de Regresión

Ahora entrenamos un árbol de regresión con los datos de training sin limitar la profundidad del árbol.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
tree_reg = DecisionTreeRegressor().fit(X_train, y_train)

Al visualizar el modelo, el ***overfitting*** resulta bastante evidente.

In [ ]:
plot_regression(X_train, y_train, X_test, y_test, tree_reg)

El overfitting es evidente: el modelo se ajusta perfectamente a los datos de training pero no generaliza bien a los datos de test. En este caso es visualmente obvio, pero también se demuestra por la gran diferencia entre el error de training y el de test. **Esto muestra claramente por qué debe separarse un conjunto de test para evaluar el rendimiento del modelo**.

Los modelos en general siempre se ajustan mejor a los datos de training que a los de test. Esto es trivial ya que el modelo fue entrenado con esos datos. Pero lo que importa es que el modelo generalice bien a datos no vistos, y eso es lo que evalúa el conjunto de test.

Es evidente cómo un decision tree tiende al overfitting mientras que un modelo lineal no. Esta es una tendencia general de los modelos **no paramétricos** frente a los modelos **paramétricos**:

- Los **modelos paramétricos** (como la regresión lineal) tienen una estructura fija con un número predeterminado de parámetros. Asumen una forma funcional específica (por ejemplo, una línea recta) y no pueden ajustarse tan estrechamente a los datos.
- Los **modelos no paramétricos** (como los decision trees) no asumen una estructura fija y pueden adaptar su complejidad a los datos, lo que los hace más flexibles pero también más propensos al overfitting.

Por ejemplo, un modelo lineal siempre será una línea recta (o un plano, o hiperplano, según la dimensionalidad de los datos), mientras que un decision tree puede ser tan complejo como se desee. Por tanto, **los modelos no paramétricos tienden más al overfitting**.

In [ ]:
print("MAE Decision Tree: ", mean_absolute_error(y_test, tree_reg.predict(X_test)))
print("MAE Decision Tree (train): ", mean_absolute_error(y_train, tree_reg.predict(X_train)))

Limitando la profundidad del árbol, podemos reducir el overfitting.

In [ ]:
tree_reg = DecisionTreeRegressor(max_depth=3).fit(X_train, y_train)
plot_regression(X_train, y_train, X_test, y_test, tree_reg)
print("MAE Decision Tree: ", mean_absolute_error(y_test, tree_reg.predict(X_test)))
print("MAE Decision Tree (train): ", mean_absolute_error(y_train, tree_reg.predict(X_train)))

La elección de la profundidad en este caso se realizó probando diferentes valores y viendo cuál funciona mejor en el conjunto de test (mejora hasta profundidad 3, pero a profundidad 4 empieza a hacer overfitting). Existen métodos más sistemáticos para esto —como la **cross-validation** y el **ajuste de hiperparámetros**— que exploraremos más adelante.

In [ ]:
plt.figure(figsize=(14, 8))

# Representar datos de training y test
plt.scatter(X, y, color='blue', label='Train')
plt.scatter(X_test, y_test, color='orange', label='Test') 

# Crear línea suave para predicciones
X_plot = pd.DataFrame(np.linspace(X.min(), X.max(), num=100), columns=X.columns)

# Entrenar modelos con distintas profundidades
tree_reg2 = DecisionTreeRegressor(max_depth=2).fit(X_train, y_train)
mae2 = round(mean_absolute_error(y_test, tree_reg2.predict(X_test)), 0)
tree_reg3 = DecisionTreeRegressor(max_depth=3).fit(X_train, y_train)
mae3 = round(mean_absolute_error(y_test, tree_reg3.predict(X_test)), 0)
tree_reg4 = DecisionTreeRegressor(max_depth=4).fit(X_train, y_train)
mae4 = round(mean_absolute_error(y_test, tree_reg4.predict(X_test)), 0)

# Representar predicciones para cada profundidad
plt.plot(X_plot, tree_reg2.predict(X_plot), color='seagreen', label=f'MAE depth=2: {mae2}', linestyle=':')
plt.plot(X_plot, tree_reg3.predict(X_plot), color='red', label=f'MAE depth=3: {mae3} (mínimo)', linewidth=2)
plt.plot(X_plot, tree_reg4.predict(X_plot), color='green', label=f'MAE depth=4: {mae4}', linestyle='-.')

# Anotar punto de overfitting
plt.annotate('Overfitting al aumentar la profundidad a 4', xy=(3.55, 128875.56), xytext=(4, 142000), 
             arrowprops=dict(facecolor='green', shrink=0.05))

plt.xlabel("Años de Experiencia")
plt.ylabel("Salario")
plt.legend()
plt.show()

## Underfitting y Overfitting

El concepto de ***underfitting*** describe un modelo demasiado simple para ajustarse bien a los datos de training. Es lo contrario del ***overfitting*** y ocurre cuando el modelo no puede capturar la complejidad de los datos.

- **Underfitting**: El modelo es demasiado simple y tiene un error alto tanto en training como en test (alto bias).
- **Overfitting**: El modelo es demasiado complejo y tiene un error bajo en los datos de training pero alto en los de test (alta varianza).
- **Buen ajuste**: El modelo captura los patrones subyacentes y generaliza bien a datos no vistos.

[![](../img/under_over_1.png)](https://ieeexplore.ieee.org/document/8405448)
[![](../img/under_over_2.png)](https://github.com/gratus907/gratus907.github.io/blob/master/images/81b7294441f2b9c96cce938661b95a1d20d22366e5c0f72e48d2c69c9c7ad7b4.png)

## Material Complementario

- [Dot CSV - ¿Las Redes Neuronales... Aprenden o Memorizan? - Overfitting y Underfitting - Parte 1](https://www.youtube.com/watch?v=7-6X3DTt3R8)